# EGU26 Hydrological GNN Experiment Notebook (v5)

This notebook tests a new end-to-end architecture:
- GCN-Embedding + FT-Transformer

Comparison target:
- current best Delta-Chl GCN + GRU baseline

New model idea:
1. apply a graph encoder at each time step
2. extract the outlet embedding sequence
3. model that sequence with an FT-Transformer block
4. predict delta-chl with the same weighted loss setting


## 1. Environment Setup


In [ ]:
import os
import sys
from pathlib import Path

if os.path.exists('/content'):
    REPO_ROOT = Path('/content/EGU26-SWAT-GNN')
    if not REPO_ROOT.exists():
        !git clone https://github.com/kona0107/EGU26-SWAT-GNN.git /content/EGU26-SWAT-GNN
    MODULE_PATH = REPO_ROOT / 'script' / 'src' / 'gnn_project'
    DATA_PATH = REPO_ROOT / 'script' / 'data'
else:
    cwd = Path.cwd()
    if (cwd / 'script' / 'src' / 'gnn_project').exists():
        REPO_ROOT = cwd
    else:
        REPO_ROOT = cwd.parents[2]
    MODULE_PATH = REPO_ROOT / 'script' / 'src' / 'gnn_project'
    DATA_PATH = REPO_ROOT / 'script' / 'data'

if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

OUTLET_CSV = DATA_PATH / 'outlet.csv'
UPSTREAM_CSV = DATA_PATH / 'upstream.csv'
RESULTS_DIR = REPO_ROOT / 'results' / 'train_v5'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT   :', REPO_ROOT)
print('DATA_PATH   :', DATA_PATH)
print('RESULTS_DIR :', RESULTS_DIR)
print('Outlet CSV  :', OUTLET_CSV.exists())
print('Upstream CSV:', UPSTREAM_CSV.exists())


In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')

if os.path.exists('/content'):
    !pip install torch_geometric -q
    !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html -q


## 2. Imports and Shared Config


In [ ]:
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import DataLoader

from data.dataset import CHL_IDX, FEATURE_DIM, OUTLET_IDX, RAW_DIM, load_real_data, prepare_and_split_data
from models.advanced_hybrid import GCNEmbeddingFTTransformerModel
from models.st_gcn import SpatioTemporalHybridGNN

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

SEED = 42
LOOKBACK = 14
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
BATCH_SIZE = 16
NUM_EPOCHS = 200
PATIENCE = 30
LR = 3e-4
HIGH_THRESHOLD_RAW = 80.0
HIGH_WEIGHT = 2.0
GRU_TEMPORAL_HIDDEN = 96
GRU_GCN_HIDDEN = 32
FT_NUM_LAYERS = 2
FT_NHEAD = 4
FT_DROPOUT = 0.1


## 3. Data Loading and Split


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

raw_features, dates = load_real_data(str(OUTLET_CSV), str(UPSTREAM_CSV))
print(f'Total dates      : {len(dates)}')
print(f'raw_features     : {raw_features.shape}  -> expected [T, 29, {RAW_DIM}]')
print(f'date range       : {dates[0]} ~ {dates[-1]}')

train_ds, val_ds, test_ds, scaler_out, scaler_target = prepare_and_split_data(
    raw_features,
    outlet_node_idx=OUTLET_IDX,
    lookback_window=LOOKBACK,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    apply_log1p=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

total_steps = len(dates)
train_end = int(total_steps * TRAIN_RATIO)
val_end = train_end + int(total_steps * VAL_RATIO)
train_dates = dates[:train_end]
val_dates = dates[train_end:val_end]
test_dates = dates[val_end:]

bx, by, _ = next(iter(train_loader))
print(f'Train / Val / Test samples: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')
print(f'Batch X shape             : {bx.shape}')
print(f'Batch y shape             : {by.shape}')


## 4. Directed Graph Definition


In [ ]:
edges_list = [
    (20, 5), (5, 23), (21, 24), (22, 24), (24, 23), (23, 25), (4, 25), (19, 25),
    (25, 27), (15, 3), (3, 27), (6, 27), (10, 27), (16, 27), (27, 0), (0, 28),
    (26, 2), (2, 1), (12, 1), (1, 28), (7, 28), (8, 28), (9, 28),
]
source = torch.tensor([edge[0] for edge in edges_list], dtype=torch.long)
target = torch.tensor([edge[1] for edge in edges_list], dtype=torch.long)
edge_index = torch.stack([source, target], dim=0).to(DEVICE)
print('edge_index shape:', edge_index.shape)


## 5. Training and Evaluation Utilities


In [ ]:
def make_weighted_mse_loss(high_threshold_raw=80.0, high_weight=2.0):
    threshold_scaled = scaler_target.transform(
        np.array([[np.log1p(high_threshold_raw)]], dtype=np.float32)
    )[0, 0]

    def weighted_mse(pred, target):
        weights = torch.ones_like(target)
        weights = torch.where(target >= threshold_scaled, high_weight, weights)
        return torch.mean(weights * (pred - target) ** 2)

    return weighted_mse


def inverse_target(arr_2d):
    return np.expm1(scaler_target.inverse_transform(arr_2d)).ravel()


def compute_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
    }


def compute_high_metrics(y_true, y_pred, threshold=80.0):
    mask = y_true >= threshold
    if mask.sum() == 0:
        return {
            'high_count': 0,
            'high_R2': np.nan,
            'high_RMSE': np.nan,
            'high_MAE': np.nan,
        }
    metrics = compute_metrics(y_true[mask], y_pred[mask])
    return {
        'high_count': int(mask.sum()),
        'high_R2': metrics['R2'],
        'high_RMSE': metrics['RMSE'],
        'high_MAE': metrics['MAE'],
    }


def build_model(config):
    family = config['model_family']

    if family == 'delta_gcn_gru':
        return SpatioTemporalHybridGNN(
            in_features=FEATURE_DIM,
            temporal_hidden=config.get('temporal_hidden', GRU_TEMPORAL_HIDDEN),
            gcn_hidden=config.get('gcn_hidden', GRU_GCN_HIDDEN),
            temporal_type='gru',
        ).to(DEVICE)

    if family == 'gcn_embedding_ft_transformer':
        return GCNEmbeddingFTTransformerModel(
            in_features=FEATURE_DIM,
            lookback_window=LOOKBACK,
            spatial_hidden=config.get('spatial_hidden', 32),
            temporal_hidden=config.get('temporal_hidden', 96),
            num_layers=config.get('ft_num_layers', FT_NUM_LAYERS),
            nhead=config.get('ft_nhead', FT_NHEAD),
            dropout=config.get('ft_dropout', FT_DROPOUT),
        ).to(DEVICE)

    raise ValueError(f"Unknown model family: {family}")


def prepare_delta_targets(x_batch, y_batch, preds):
    last_outlet_chl = x_batch[:, -1, OUTLET_IDX, CHL_IDX].unsqueeze(-1)
    target = y_batch.view_as(preds) - last_outlet_chl
    pred_absolute = preds + last_outlet_chl
    return target, pred_absolute


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    y_true_scaled_all = []
    y_pred_scaled_all = []

    for x_batch, y_batch, _ in loader:
        x_batch = x_batch.to(DEVICE)
        preds = model(x_batch, edge_index, outlet_node_idx=OUTLET_IDX)
        _, pred_absolute = prepare_delta_targets(x_batch, y_batch.to(DEVICE), preds)
        y_true_scaled_all.append(y_batch.numpy())
        y_pred_scaled_all.append(pred_absolute.cpu().numpy())

    y_true_scaled = np.concatenate(y_true_scaled_all, axis=0)
    y_pred_scaled = np.concatenate(y_pred_scaled_all, axis=0)
    y_true_raw = inverse_target(y_true_scaled)
    y_pred_raw = inverse_target(y_pred_scaled)
    return y_true_raw, y_pred_raw


def train_and_evaluate(config):
    set_seed(SEED)
    model = build_model(config)
    criterion = make_weighted_mse_loss(
        high_threshold_raw=config.get('high_threshold', HIGH_THRESHOLD_RAW),
        high_weight=config.get('high_weight', HIGH_WEIGHT),
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=config.get('lr', LR))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

    best_val_loss = float('inf')
    best_state = None
    patience_count = 0
    history = []

    start_time = time.time()
    for epoch in range(1, config.get('num_epochs', NUM_EPOCHS) + 1):
        model.train()
        train_loss_sum = 0.0

        for x_batch, y_batch, _ in train_loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            preds = model(x_batch, edge_index, outlet_node_idx=OUTLET_IDX)
            target_for_loss, _ = prepare_delta_targets(x_batch, y_batch, preds)
            loss = criterion(preds, target_for_loss)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * len(x_batch)

        train_loss = train_loss_sum / len(train_loader.dataset)

        model.eval()
        val_loss_sum = 0.0
        with torch.no_grad():
            for x_batch, y_batch, _ in val_loader:
                x_batch = x_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
                preds = model(x_batch, edge_index, outlet_node_idx=OUTLET_IDX)
                target_for_loss, _ = prepare_delta_targets(x_batch, y_batch, preds)
                loss = criterion(preds, target_for_loss)
                val_loss_sum += loss.item() * len(x_batch)

        val_loss = val_loss_sum / len(val_loader.dataset)
        scheduler.step(val_loss)

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'lr': optimizer.param_groups[0]['lr'],
        })

        if epoch == 1 or epoch % 20 == 0:
            print(f"[{config['name']}] epoch={epoch:3d} train={train_loss:.4f} val={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= config.get('patience', PATIENCE):
                print(f"[{config['name']}] Early stopping at epoch {epoch}, best val={best_val_loss:.4f}")
                break

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)

    tr_true, tr_pred = collect_predictions(model, train_loader)
    va_true, va_pred = collect_predictions(model, val_loader)
    te_true, te_pred = collect_predictions(model, test_loader)

    train_metrics = compute_metrics(tr_true, tr_pred)
    val_metrics = compute_metrics(va_true, va_pred)
    test_metrics = compute_metrics(te_true, te_pred)
    val_high = compute_high_metrics(va_true, va_pred, threshold=config.get('high_threshold', HIGH_THRESHOLD_RAW))
    test_high = compute_high_metrics(te_true, te_pred, threshold=config.get('high_threshold', HIGH_THRESHOLD_RAW))

    return {
        'name': config['name'],
        'model_family': config['model_family'],
        'temporal_hidden': config.get('temporal_hidden', np.nan),
        'gcn_hidden': config.get('gcn_hidden', np.nan),
        'spatial_hidden': config.get('spatial_hidden', np.nan),
        'lr': config.get('lr', LR),
        'train_R2': train_metrics['R2'],
        'train_RMSE': train_metrics['RMSE'],
        'train_MAE': train_metrics['MAE'],
        'val_R2': val_metrics['R2'],
        'val_RMSE': val_metrics['RMSE'],
        'val_MAE': val_metrics['MAE'],
        'val_high_MAE': val_high['high_MAE'],
        'val_high_RMSE': val_high['high_RMSE'],
        'test_R2': test_metrics['R2'],
        'test_RMSE': test_metrics['RMSE'],
        'test_MAE': test_metrics['MAE'],
        'test_high_MAE': test_high['high_MAE'],
        'test_high_RMSE': test_high['high_RMSE'],
        'best_val_loss': best_val_loss,
        'elapsed_sec': time.time() - start_time,
    }, history_df


## 6. Experiment Definitions


In [ ]:
experiment_configs = [
    {
        'name': 'delta_gru_best',
        'model_family': 'delta_gcn_gru',
        'temporal_hidden': 96,
        'gcn_hidden': 32,
        'lr': 3e-4,
        'high_threshold': 80.0,
        'high_weight': 2.0,
    },
    {
        'name': 'gcn_embed_ft_s32_t96',
        'model_family': 'gcn_embedding_ft_transformer',
        'spatial_hidden': 32,
        'temporal_hidden': 96,
        'lr': 3e-4,
        'high_threshold': 80.0,
        'high_weight': 2.0,
    },
    {
        'name': 'gcn_embed_ft_s48_t96',
        'model_family': 'gcn_embedding_ft_transformer',
        'spatial_hidden': 48,
        'temporal_hidden': 96,
        'lr': 3e-4,
        'high_threshold': 80.0,
        'high_weight': 2.0,
    },
    {
        'name': 'gcn_embed_ft_s32_t128',
        'model_family': 'gcn_embedding_ft_transformer',
        'spatial_hidden': 32,
        'temporal_hidden': 128,
        'lr': 3e-4,
        'high_threshold': 80.0,
        'high_weight': 2.0,
    },
]

pd.DataFrame(experiment_configs)


## 7. Run Experiments


In [ ]:
    all_results = []
    history_map = {}

    for idx, config in enumerate(experiment_configs, start=1):
        print(f"
===== [{idx}/{len(experiment_configs)}] RUN: {config['name']} =====")
        result, history_df = train_and_evaluate(config)
        all_results.append(result)
        history_map[config['name']] = history_df
        history_df.to_csv(RESULTS_DIR / f"history_{config['name']}.csv", index=False)

    results_df = pd.DataFrame(all_results).sort_values(
        ['val_R2', 'val_high_MAE', 'test_R2'],
        ascending=[False, True, False],
    )
    results_df.to_csv(RESULTS_DIR / 'summary_train_v5.csv', index=False)
    display(results_df)
    print('Saved summary to:', RESULTS_DIR / 'summary_train_v5.csv')


## 8. Comparison Views


In [ ]:
summary_cols = [
    'name', 'model_family', 'spatial_hidden', 'temporal_hidden', 'lr',
    'train_R2', 'val_R2', 'test_R2',
    'val_high_MAE', 'test_high_MAE',
    'val_high_RMSE', 'test_high_RMSE',
    'best_val_loss', 'elapsed_sec',
]

print('=== Ranked by val_R2, then val_high_MAE ===')
display(results_df[summary_cols])

print('=== Ranked by test_high_MAE ===')
display(results_df.sort_values(['test_high_MAE', 'val_R2', 'test_R2'], ascending=[True, False, False])[summary_cols])


## 9. Loss Curves


In [ ]:
plt.figure(figsize=(14, 5))
for name, history_df in history_map.items():
    plt.plot(history_df['epoch'], history_df['val_loss'], label=f'{name} val')
plt.title('Validation Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
